- Khởi tạo tokenizer

In [5]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Special tokens:", tokenizer.special_tokens_map)

Tokenizer: BertTokenizer
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [6]:
text = "The cats are playing."

tokens = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt",
)

print("tokens.keys():", list(tokens.keys()))
print("input_ids:", tokens["input_ids"])
print("attention_mask:", tokens["attention_mask"])

token_ids = tokens["input_ids"][0].tolist()
token_sequence = tokenizer.convert_ids_to_tokens(token_ids)
print("token sequence:", token_sequence)
     

print("Special token IDs:")
for name, token in tokenizer.special_tokens_map.items():
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"{name}: {token!r} -> {token_id}")

print("\nToken and mask:")
for token, token_id, mask_value in zip(
    token_sequence,
    tokens["input_ids"][0].tolist(),
    tokens["attention_mask"][0].tolist(),
):
    print(f"{token:>12} | id={token_id:>6} | attention={mask_value}")

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: tensor([[ 101, 1996, 8870, 2024, 2652, 1012,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
token sequence: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Special token IDs:
unk_token: '[UNK]' -> 100
sep_token: '[SEP]' -> 102
pad_token: '[PAD]' -> 0
cls_token: '[CLS]' -> 101
mask_token: '[MASK]' -> 103

Token and mask:
       [CLS] | id=   101 | attention=1
         the | id=  1996 | attention=1
        cats | id=  8870 | attention=1
         are | id=  2024 | attention=1
     playing | id=  2652 | attention=1
           . | id=  1012 | attention=1
       [SEP] | id=   102 | attention=1
       [PAD] | id=     0 | attention=0
       [PAD] | id=     0 | attention=0
       [PAD] | id=     0 | attention=0
       [PAD] | id=     

- Trong attention mask, 1 là vị trí token hợp lệ; 0 là vị trí padding. Các token đặc biệt thường gặp là [CLS] (đại diện sequence), [SEP] (phân tách sequence), [PAD] (padding), [MASK] (MLM) và [UNK] (token không nhận diện được, tùy tokenizer).

In [7]:
words = text.rstrip(".").split() + ["."]
active_tokens = [
    token
    for token, mask_value in zip(
        token_sequence,
        tokens["attention_mask"][0].tolist(),
    )
    if mask_value == 1
]

print("Words:", words)
print("Số words:", len(words))
print("Số tokens hợp lệ:", len(active_tokens))
print("Số subword tokens trong nội dung:", sum(
    1 for token in active_tokens if not token.startswith("[")
))
print("Lưu ý: một word có thể tách thành nhiều subword, ví dụ các token có tiền tố ##.")

Words: ['The', 'cats', 'are', 'playing', '.']
Số words: 5
Số tokens hợp lệ: 7
Số subword tokens trong nội dung: 5
Lưu ý: một word có thể tách thành nhiều subword, ví dụ các token có tiền tố ##.


- Biểu diễn [CLS] trong Transformer

In [8]:
import torch

model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

with torch.no_grad():
    outputs = model(**tokens)

last_hidden_state = outputs.last_hidden_state
cls_representation = last_hidden_state[:, 0, :]

print("last_hidden_state shape:", tuple(last_hidden_state.shape))
print("[CLS] representation shape:", tuple(cls_representation.shape))

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 171.01it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: (1, 16, 768)
[CLS] representation shape: (1, 768)
